# Pandas `.soc` accessor — 5-line SOC validation

If you already have your data in a `pd.Series`, the `.soc` accessor gives you SOC validation directly: no need to convert to a numpy array, no need to learn the FitResult API.

```python
import pandas as pd, soc_pipeline
s = pd.Series(my_event_sizes)
s.soc.fit_alpha()                    # -> float
s.soc.validate(expected_band=(2.5, 3.5))  # -> Verdict
s.soc.is_pass(expected_band=(2.5, 3.5))   # -> bool
```

In [ ]:
import sys
from pathlib import Path

_here = Path('.').resolve()
for _ in range(6):
    _candidate = _here / 'packages' / 'soc-pipeline' / 'src'
    if _candidate.is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
    _here = _here.parent
for _mod in list(sys.modules):
    if _mod == 'soc_pipeline' or _mod.startswith('soc_pipeline.'):
        del sys.modules[_mod]

import json
import numpy as np
import pandas as pd
import soc_pipeline  # noqa: F401 — registers the accessor
print('soc_pipeline version:', soc_pipeline.__version__)

## 1. Load earthquake magnitudes into a Series

In [ ]:
REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'dataset' / 'v1').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
catalog = REPO_ROOT / 'dataset/v1/systems/01_earthquake/data/catalog.jsonl'

records = []
if catalog.exists():
    with catalog.open() as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if r.get('mag') is not None:
                records.append(float(r['mag']))
    s = pd.Series(records, name='earthquake_magnitudes')
else:
    # Fallback to synthetic Pareto if the bundled catalog isn't checked out (LFS pointer).
    rng = np.random.default_rng(0)
    s = pd.Series(rng.pareto(1.5, 5000) + 1.0, name='synthetic_pareto')

print(s.describe())

## 2. Fit α directly

In [ ]:
alpha = s.soc.fit_alpha()
print(f'alpha = {alpha:.3f}')

## 3. Full validation with expected band

The Gutenberg-Richter literature pins earthquake-magnitude α in roughly (2.5, 3.5). We pass that as the prediction band.

In [ ]:
v = s.soc.validate(expected_band=(2.5, 3.5), n_boot=50)
print('verdict:           ', v.verdict)
print('underlying verdict:', v.underlying_verdict)
print('alpha:             ', round(v.alpha, 3) if v.alpha is not None else None)
if v.alpha_ci_low is not None:
    print(f'95% CI:            [{v.alpha_ci_low:.3f}, {v.alpha_ci_high:.3f}]')
print('xmin:              ', round(v.xmin, 3) if v.xmin is not None else None)
print('n_tail / n_total:  ', f'{v.n_tail} / {v.n_total}')
print('in_band:           ', v.in_band)

## 4. Quick PASS/FAIL gate

For pipelines / CI checks, `is_pass()` collapses everything into a boolean.

In [ ]:
print('is_pass (2.5–3.5):', s.soc.is_pass(expected_band=(2.5, 3.5)))
print('is_pass (5.0–6.0):', s.soc.is_pass(expected_band=(5.0, 6.0)))  # absurd band -> False

## 5. JSON-friendly record

`Verdict.to_dict()` drops the heavyweight FitResult / BootstrapResult objects so you can `json.dumps()` the result straight into a logging table.

In [ ]:
import json as _json
print(_json.dumps(v.to_dict(), indent=2, default=str))